# Macaulay Equity Duration (FY1-FY3 + Transition + Terminal)

This notebook computes firm-level Macaulay equity duration from analyst EPS forecasts:
- Explicit phase: FY1-FY3
- Transition phase: years 4-10 with linearly fading growth
- Terminal phase: perpetual growth with \(g_\infty = 2\%\)


## 1) Load data


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# Project-style storage layout (consistent with other notebooks)
BASE_DIR = Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data')
DATA_DIR = BASE_DIR / 'intermediate'

INPUT_PATH = DATA_DIR / 'euro500_macaulay.parquet'
OUTPUT_PATH = DATA_DIR / 'EQDuration_Macaulay.parquet'

df = pd.read_parquet(INPUT_PATH)
print(f'Loaded rows: {len(df):,}')
print(f'Loaded columns: {len(df.columns):,}')
print(f'Input file: {INPUT_PATH}')



Loaded rows: 56,500
Loaded columns: 40
Input file: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_macaulay.parquet


In [2]:
# Required non-missing fields for this model
required_cols = [
    'PriceClose',
    'EPS_FY1',
    'EPS_FY2',
    'EPS_FY3',
    'market_risk_free_rate_annual',
]

# Step 10 cleaning rule (applied before valuation): remove negative EPS_FY1 and PriceClose
df = df.dropna(subset=required_cols).copy()
df = df[(df['EPS_FY1'] >= 0) & (df['PriceClose'] > 0)].copy()

print(f'Rows after required-field + sign filters: {len(df):,}')


Rows after required-field + sign filters: 21,896


## 2) Discount rate

Use a constant equity risk premium of 5%:
$$
r = rf + 0.05
$$


In [3]:
ERP = 0.05
G_INF = 0.02

# Convert risk-free rate to numeric and construct discount rate
# Assumption: market_risk_free_rate_annual is already in decimal terms.
df['rf'] = pd.to_numeric(df['market_risk_free_rate_annual'], errors='coerce')
df['r'] = df['rf'] + ERP

# Keep only economically valid rows for terminal value denominator (r - g_inf > 0)
df = df[df['r'] > G_INF].copy()
print(f'Rows after requiring r > g_inf: {len(df):,}')


Rows after requiring r > g_inf: 21,896


## 3) LTG preparation

LTG is provided in percent. Convert to decimal and clip:
$$
g_0 = \mathrm{clip}(LTG/100, -0.02, 0.15)
$$


In [4]:
df['LTG_decimal'] = pd.to_numeric(df['LTG'], errors='coerce') / 100.0
df['g0'] = df['LTG_decimal'].clip(lower=-0.02, upper=0.15)

## 4) Explicit cashflows (years 1-3)


In [5]:
df['C1'] = pd.to_numeric(df['EPS_FY1'], errors='coerce')
df['C2'] = pd.to_numeric(df['EPS_FY2'], errors='coerce')
df['C3'] = pd.to_numeric(df['EPS_FY3'], errors='coerce')

## 5) Transition growth path (years 4-10)

Growth fades linearly from $g_0$ in year 4 to $g_\infty=2\%$ in year 10:
$$
g_t = g_0 + \frac{t-4}{10-4}(g_\infty - g_0),\quad t=4,\ldots,10
$$
Then recursively:
$$
C_t = C_{t-1}(1+g_t)
$$


In [6]:
# Build year-specific growth rates and recursive cashflows
for t in range(4, 11):
    df[f'g_{t}'] = df['g0'] + ((t - 4) / 6.0) * (G_INF - df['g0'])

df['C4'] = df['C3'] * (1.0 + df['g_4'])
for t in range(5, 11):
    df[f'C{t}'] = df[f'C{t-1}'] * (1.0 + df[f'g_{t}'])


## 6) Terminal value

$$
C_{11} = C_{10}(1+g_\infty),\quad TV_{10}=\frac{C_{11}}{r-g_\infty}
$$


In [7]:
df['C11'] = df['C10'] * (1.0 + G_INF)
df['TV10'] = df['C11'] / (df['r'] - G_INF)


## 7) Present value of cashflows


In [8]:
# Discount factors and PVs for years 1..10
for t in range(1, 11):
    df[f'PV_{t}'] = df[f'C{t}'] / np.power(1.0 + df['r'], t)

df['PV_TV'] = df['TV10'] / np.power(1.0 + df['r'], 10)

pv_cols = [f'PV_{t}' for t in range(1, 11)]
df['PV_total'] = df[pv_cols].sum(axis=1) + df['PV_TV']


## 8) Macaulay duration

Numerator:
$$
\sum_{t=1}^{10} t\,PV_t + T_{TV}\,PV_{TV}
$$
with terminal maturity:
$$
T_{TV} = 10 + \frac{1+r}{r-g_\infty}
$$

Duration:
$$
D = \frac{	ext{Numerator}}{	ext{PriceClose}}
$$


In [9]:
# Duration numerator for explicit years
duration_numerator = np.zeros(len(df), dtype=float)
for t in range(1, 11):
    duration_numerator += t * df[f'PV_{t}'].to_numpy()

# Add terminal contribution
df['T_TV'] = 10.0 + (1.0 + df['r']) / (df['r'] - G_INF)
duration_numerator += (df['T_TV'] * df['PV_TV']).to_numpy()

df['duration_macaulay'] = duration_numerator / df['PriceClose']


## 9) Diagnostics


In [10]:
# 1) Terminal value share
df['TV_share'] = df['PV_TV'] / df['PV_total']

# 2) Duration summary statistics
duration_summary = df['duration_macaulay'].describe()
print('Duration summary:')
print(duration_summary)

# 3) Cross-sectional std by date
cs_std_by_date = df.groupby('date', dropna=False)['duration_macaulay'].std()
print('Cross-sectional duration std (first 10 dates):')
print(cs_std_by_date.head(10))

# 4) Correlation with LTG (decimal LTG)
corr_duration_ltg = df['duration_macaulay'].corr(df['LTG_decimal'])
print(f'Correlation duration vs LTG_decimal: {corr_duration_ltg:.6f}')


Duration summary:
count         13797.0
mean       239.966575
std       1726.713445
min       -948.757903
25%         83.325002
50%        153.325012
75%        265.254426
max      180536.35629
Name: duration_macaulay, dtype: Float64
Cross-sectional duration std (first 10 dates):
date
2003-12-31    109.435937
2004-03-31    170.065511
2004-06-30    152.122344
2004-09-30    145.199867
2004-12-31    137.555179
2005-03-31     44.875224
2005-06-30    115.901127
2005-09-30    329.705396
2005-12-30     95.609727
2006-03-31    103.053147
Name: duration_macaulay, dtype: Float64
Correlation duration vs LTG_decimal: 0.021845


## 10) Winsorize duration (1% / 99%)


In [11]:
q01 = df['duration_macaulay'].quantile(0.01)
q99 = df['duration_macaulay'].quantile(0.99)
df['duration_macaulay'] = df['duration_macaulay'].clip(lower=q01, upper=q99)

print(f'Winsorization bounds: [{q01:.6f}, {q99:.6f}]')
print(df['duration_macaulay'].describe())


Winsorization bounds: [17.695966, 1200.353991]
count        13797.0
mean      209.451502
std       198.270394
min        17.695966
25%        83.325002
50%       153.325012
75%       265.254426
max      1200.353991
Name: duration_macaulay, dtype: Float64


## 11) Output


In [12]:
# Keep all original columns and add key output fields
output_cols = list(df.columns)

# Persist result to intermediate folder
DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Rows saved: {len(df):,}')
print('Added fields include duration_macaulay and TV_share.')



Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/EQDuration_Macaulay.parquet
Rows saved: 21,896
Added fields include duration_macaulay and TV_share.
